In [0]:
%sql

-- Criando a camada Gold, gerando uma tabela juntando todos os dados

CREATE OR REPLACE TABLE delta.`dbfs:/FileStore/project/olist/gold/sales` 
USING DELTA PARTITIONED BY (estadoCliente) 
(
  SELECT
    CASE
      WHEN orders.orderStatus = 'shipped' THEN 'enviado'
      WHEN orders.orderStatus = 'canceled' THEN 'cancelado'
      WHEN orders.orderStatus = 'invoiced' THEN 'faturado'
      WHEN orders.orderStatus = 'created' THEN 'criado'
      WHEN orders.orderStatus = 'delivered' THEN 'entregue'
      WHEN orders.orderStatus = 'unavailable' THEN 'indisponível'
      WHEN orders.orderStatus = 'processing' THEN 'em processamento'
      WHEN orders.orderStatus = 'approved' THEN 'aprovado'
    END AS statusDoPedido,
    orders.orderPurchaseTimestamp AS horaCompraPedido,
    orders.orderApprovedAt AS horaPedidoAprovado,
    orders.orderEstimatedDeliveryDate AS dataEstimadaEntrega,
    DATEDIFF(
      orders.orderEstimatedDeliveryDate,
      orders.orderApprovedAt
    ) AS dataEntregaEmDias,
    order_reviews.reviewScore AS notaProduto,
    order_reviews.reviewAnswerTimestamp AS dataComentarioSobreProduto,
    CASE
      WHEN order_payments.paymentType = 'credit_card' THEN 'cartao_de_credito'
      WHEN order_payments.paymentType = 'boleto' THEN 'boleto'
      WHEN order_payments.paymentType = 'not_defined' THEN 'não_definido'
      WHEN order_payments.paymentType = 'voucher' THEN 'voucher'
      WHEN order_payments.paymentType = 'debit_card' THEN 'cartao_de_debito'
    END AS meioDePagamento,
    order_payments.paymentInstallments AS parcelamento,
    order_payments.paymentValue AS valorPago,
    customers.customerCity AS cidadeCliente,
    customers.customerState AS estadoCliente
  FROM
    delta.`dbfs:/FileStore/project/olist/silver/orders` orders
    LEFT JOIN delta.`dbfs:/FileStore/project/olist/silver/order_payments` 
        order_payments ON order_payments.orderId = orders.orderId
    LEFT JOIN delta.`dbfs:/FileStore/project/olist/silver/order_reviews` 
        order_reviews ON order_reviews.orderId = orders.orderId
    LEFT JOIN delta.`dbfs:/FileStore/project/olist/silver/customers` 
        customers ON customers.customerId = orders.customerId
)

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
  estadoCliente AS estados,
  meioDePagamento AS `meio de pagamento`,
  count(*) AS `percentual`
FROM
  delta.`dbfs:/FileStore/project/olist/gold/sales`
WHERE
  meioDePagamento IS NOT NULL
  AND YEAR(horaPedidoAprovado) IS NOT NULL
  AND statusDoPedido = "entregue"
GROUP BY
  estadoCliente,
  meioDePagamento

estados,meio de pagamento,percentual
PR,boleto,1091
MG,cartao_de_credito,8905
BA,cartao_de_debito,51
SC,cartao_de_credito,2660
RJ,cartao_de_debito,181
MG,boleto,2260
RJ,cartao_de_credito,9934
SP,cartao_de_debito,740
SC,boleto,819
PR,cartao_de_debito,75


In [0]:
%sql 

SELECT
  T.estadoCliente AS estados,
  T.diasEntrega AS `média de dias para entrega de produto`
FROM
  (
    SELECT
      estadoCliente,
      ROUND(AVG(dataEntregaEmDias), 0) AS DiasEntrega
    FROM
      delta.`dbfs:/FileStore/project/olist/gold/sales`
    WHERE
      meioDePagamento IS NOT NULL
      AND YEAR(horaPedidoAprovado) IS NOT NULL
      AND statusDoPedido <> "cancelado"
    GROUP BY
      estadoCliente
  ) AS T

estados,média de dias para entrega de produto
SC,26.0
SP,19.0
RS,29.0
MG,25.0
BA,29.0
RJ,27.0
PR,25.0
GO,27.0
MT,32.0
ES,26.0


In [0]:
%sql
SELECT
    estadoCliente as `Estado`,
    Year(horaPedidoAprovado) as `Ano`,
    Count(*) as `Numero de Vendas`
FROM
    delta.`dbfs:/FileStore/project/olist/gold/sales`
WHERE
    statusDoPedido = "entregue"
And
    Year(horaPedidoAprovado) = "2018"
GROUP BY
    Estado, Ano

Estado,Ano,Numero de Vendas
SP,2018,24224
PR,2018,2835
RS,2018,2850
RJ,2018,6687
MG,2018,6320
BA,2018,1843
SC,2018,1948
DF,2018,1234
PE,2018,883
GO,2018,1089


In [0]:
%sql

SELECT
  ROUND(SUM(valorPago)/Count(*), 2) as `Ticket Médio`,
  Month(horaPedidoAprovado)
FROM
  delta.`dbfs:/FileStore/project/olist/gold/sales`
WHERE
  statusDoPedido = "entregue" AND Year(horaPedidoAprovado) = "2017"
GROUP BY
  Month(horaPedidoAprovado)

Ticket Médio,month(horaPedidoAprovado)
146.96,12
158.2,1
148.18,6
151.59,3
149.5,5
157.09,9
160.77,4
146.23,8
136.16,7
162.0,10
